# Assignment 4 — Naive Bayes Classification of Student Outcomes (Pass/Fail)
**Dataset:** UCI Student Performance (`student-por.csv`)  
**Target:** `Pass = 1 if G3 >= 10 else 0` (early prediction: G1/G2/G3 excluded from features)

**Aim:** Train Gaussian Naive Bayes with a proper ColumnTransformer+Pipeline, stratified split, and evaluate with accuracy/precision/recall/F1, classification report, confusion matrix, ROC curve and ROC-AUC.


## 1. Library imports

In [1]:
import io, os
import urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, roc_auc_score)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset loading (official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
with zf.open("student-por.csv") as f:
    df = pd.read_csv(f, sep=";")
print("Shape:", df.shape)

Shape: (649, 33)


## 3. Dataset preview

In [3]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
print("G3 describe:"); print(df["G3"].describe())
print("\nMissing total:", int(df.isnull().sum().sum()))

G3 describe:
count    649.000000
mean      11.906009
std        3.230656
min        0.000000
25%       10.000000
50%       12.000000
75%       14.000000
max       19.000000
Name: G3, dtype: float64

Missing total: 0


## 4. Data preprocessing — features (exclude G1/G2/G3) + Pass/Fail target

In [5]:
df["Pass"] = (df["G3"] >= 10).astype(int)
print("Class distribution:"); print(df["Pass"].value_counts())
print((df["Pass"].value_counts(normalize=True)*100).round(2))
X_raw = df.drop(columns=["G1", "G2", "G3", "Pass"])
y = df["Pass"].to_numpy()
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(exclude=["object"]).columns.tolist()
print(f"\nFeatures: {X_raw.shape[1]} (numeric={len(num_cols)}, categorical={len(cat_cols)})")
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])

Class distribution:
Pass
1    549
0    100
Name: count, dtype: int64
Pass
1    84.59
0    15.41
Name: proportion, dtype: float64

Features: 30 (numeric=13, categorical=17)


C:\Users\ns479\AppData\Local\Temp\ipykernel_14988\1170947305.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()


## 5. Model implementation — stratified split + GaussianNB pipeline

In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {X_train_raw.shape}, Test: {X_test_raw.shape}")
print("Train positive rate:", y_train.mean().round(4), "| Test positive rate:", y_test.mean().round(4))
pipe = Pipeline([("prep", preprocess), ("nb", GaussianNB())])
pipe.fit(X_train_raw, y_train)
print("Pipeline steps:", list(pipe.named_steps.keys()))

Train: (519, 30), Test: (130, 30)
Train positive rate: 0.8459 | Test positive rate: 0.8462


Pipeline steps: ['prep', 'nb']


## 6. Evaluation — accuracy, precision, recall, F1, report

In [7]:
y_pred = pipe.predict(X_test_raw)
y_proba = pipe.predict_proba(X_test_raw)[:, 1]
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_proba)
print(f"Accuracy : {acc:.4f}\nPrecision: {prec:.4f}\nRecall   : {rec:.4f}\nF1-score : {f1:.4f}\nROC-AUC  : {auc:.4f}")
print("\nClassification report:"); print(classification_report(y_test, y_pred, target_names=["Fail(0)", "Pass(1)"]))
pd.DataFrame([{"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc}]).round(4).to_csv("outputs/metrics.csv", index=False)
print("Saved: outputs/metrics.csv")

Accuracy : 0.6308
Precision: 0.8690
Recall   : 0.6636
F1-score : 0.7526
ROC-AUC  : 0.4950

Classification report:
              precision    recall  f1-score   support

     Fail(0)       0.20      0.45      0.27        20
     Pass(1)       0.87      0.66      0.75       110

    accuracy                           0.63       130
   macro avg       0.53      0.56      0.51       130
weighted avg       0.77      0.63      0.68       130

Saved: outputs/metrics.csv


## 7. Tables and graphs — confusion matrix + ROC curve

In [8]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix [[TN FP],[FN TP]]:"); print(cm)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred Fail", "Pred Pass"], yticklabels=["True Fail", "True Pass"])
axes[0].set_title(f"Confusion matrix (acc={acc:.3f})")
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f"GaussianNB (AUC={auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", label="Chance")
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].set_title("ROC curve"); axes[1].legend()
plt.tight_layout(); plt.savefig("outputs/confusion_roc.png", dpi=150); plt.close()
print("Saved: outputs/confusion_roc.png")

Confusion matrix [[TN FP],[FN TP]]:
[[ 9 11]
 [37 73]]


Saved: outputs/confusion_roc.png


## 8. Result and conclusion
- Stratified 80/20 split (`random_state=42`), GaussianNB inside a ColumnTransformer pipeline.
- Metrics (accuracy/precision/recall/F1/ROC-AUC), classification report, confusion matrix and ROC curve are computed at runtime and saved to `outputs/`.
